In [ ]:
from bs4 import BeautifulSoup
import re

In [ ]:
with open('Data/Cau1/viec_lam_p1.htm', encoding='utf-8') as f:
    soup1 = f.read()

with open('Data/Cau1/viec_lam_p2.htm', encoding='utf-8') as f:
    soup2 = f.read()

In [ ]:
soup1 = BeautifulSoup(soup1, 'html5lib')
soup2 = BeautifulSoup(soup2, 'html5lib')

In [ ]:
soup1

In [ ]:
data = []

job_cards_1 = soup1.find_all('div', class_="job-card")

for i in job_cards_1:
    ma_tin = i['data-job']
    vi_tri = i.find('h4', class_="job-title").text.strip()
    cong_ty = i.find('p', class_="company").text.strip()
    muc_luong_tmp = i.find('p', class_="pay").text.strip()
    muc_luong = re.search(r"Lương: (.+)", muc_luong_tmp)[1]
    han_nop_tmp = i.find('span', class_="deadline").text.strip()
    han_nop = re.search(r"Hạn nộp: (.+)", han_nop_tmp)[1]
    
    try:
        phuc_loi = i.find('ul', class_="perks").get_text(separator=" | ")	
    except:
        phuc_loi = None
    
    data.append({
        'ma_tin':ma_tin,
        'vi_tri':vi_tri,
        'cong_ty': cong_ty,
        'muc_luong': muc_luong,
        'han_nop':han_nop,
        'phuc_loi': phuc_loi
    })


In [ ]:
job_cards_2 = soup2.find_all('div', class_="job-card")

for i in job_cards_2:
    ma_tin = i['data-job']
    vi_tri = i.find('h4', class_="job-title").text.strip()
    cong_ty = i.find('p', class_="company").text.strip()
    muc_luong_tmp = i.find('p', class_="pay").text.strip()
    muc_luong = re.search(r"Lương: (.+)", muc_luong_tmp)[1]
    han_nop_tmp = i.find('span', class_="deadline").text.strip()
    han_nop = re.search(r"Hạn nộp: (.+)", han_nop_tmp)[1]
    
    try:
        phuc_loi = i.find('ul', class_="perks").get_text(separator=" | ")	
    except:
        phuc_loi = None
    
    data.append({
        'ma_tin':ma_tin,
        'vi_tri':vi_tri,
        'cong_ty': cong_ty,
        'muc_luong': muc_luong,
        'han_nop':han_nop,
        'phuc_loi': phuc_loi
    })

In [ ]:
len(data)

In [ ]:
import pandas as pd
viec_lam_df = pd.DataFrame(data, columns=['ma_tin', 'vi_tri', 'cong_ty', 'muc_luong', 'han_nop', 'phuc_loi'])

In [ ]:
viec_lam_df

In [ ]:
import os
# os.makedirs('Cau1')

viec_lam_df.to_csv('Cau1/tin_tuyen_dung_goc.csv', index=False)

In [ ]:
# 1b



a = viec_lam_df['muc_luong'].str.split()
for i in range(len(a)):
    if len(a[i])==4:
        viec_lam_df.loc[i, 'luong_min_trieu'] = float(a[i][0].replace(',', '.'))
        viec_lam_df.loc[i, 'luong_max_trieu'] = float(a[i][2].replace(',', '.'))
    elif len(a[i])==2 and 'Thỏa' not in a[i]:
        viec_lam_df.loc[i, 'luong_min_trieu'] = float(a[i][0].replace(',', '.'))
        viec_lam_df.loc[i, 'luong_max_trieu'] = float(a[i][0].replace(',', '.'))
    else:
        viec_lam_df.loc[i, 'luong_min_trieu'] = None
        viec_lam_df.loc[i, 'luong_max_trieu'] = None
        

In [ ]:
viec_lam_df['han_nop'] = pd.to_datetime(viec_lam_df['han_nop'])

In [ ]:
viec_lam_df

In [ ]:
# 2

In [ ]:
y_kien = pd.read_csv('Data/Cau2/y_kien_ktx.csv')
y_kien

In [ ]:
tu_dien = pd.read_csv('Data/Cau2/tu_dien_ktx.csv')
tu_dien

In [ ]:
bieu_tuong = pd.read_csv('Data/Cau2/bieu_tuong.csv')
bieu_tuong

In [ ]:
# a
y_kien['van_ban_1'] = y_kien['noi_dung_goc'].str.lower()

def remove_html(text):
    return re.sub(r'<[^>]+[>}]+', '', text)

def remove_brackets(text):
    return re.sub(r'{[^}]+}', '', text)

def remove_tags(text):
    tmp = re.sub(r'\.b', '', text)
    return re.sub(r'track\(\)', '', tmp)

def remove_digits(text):
    return re.sub(r'\d+', '', text)

def remove_urls(text):
    tmp =  re.sub(r'https?://\S+', '', text)
    res = re.sub(r'www\.\S+', '', tmp)
    return res

def normalize_space(text):
    return re.sub(r'\s+', ' ', text)


y_kien['van_ban_1'] = y_kien['van_ban_1'].apply(remove_html)
y_kien['van_ban_1'] = y_kien['van_ban_1'].apply(remove_brackets)
y_kien['van_ban_1'] = y_kien['van_ban_1'].apply(remove_tags)
y_kien['van_ban_1'] = y_kien['van_ban_1'].apply(remove_urls)
y_kien['van_ban_1'] = y_kien['van_ban_1'].apply(remove_digits)
y_kien['van_ban_1'] = y_kien['van_ban_1'].apply(normalize_space)

y_kien['tach_cau'] = y_kien['van_ban_1'].str.split(re.compile(r'[!|?|.]'))

for i in range(len(y_kien)):
    cnt=0
    for j in y_kien.loc[i, 'tach_cau']:
        if j != '':
            cnt+=1
    
    y_kien.loc[i, 'so_cau'] = cnt


In [ ]:
# os.makedirs('Cau2')
y_kien[['ma_phan_hoi', 'van_ban_1', 'so_cau']].to_csv('Cau2/lam_sach_1.csv', index=False)

In [ ]:
# b
lam_sach_1 = pd.read_csv('Cau2/lam_sach_1.csv')

emoji_dict = bieu_tuong.set_index(bieu_tuong.columns[0]).to_dict()[bieu_tuong.columns[1]]
emoji_dict

In [ ]:
td_dict = tu_dien.set_index(tu_dien.columns[0]).to_dict()[tu_dien.columns[1]]
td_dict

In [ ]:
def remove_punctuation(text):
    res = re.sub(r'[^\w\s]*', '', text)
    return res

def replace_emojis_and_words(text, emoji_dict):
    for k, v in emoji_dict.items():
        text = re.sub(re.escape(k), v, text, flags=re.IGNORECASE)
    return text

lam_sach_1['van_ban_2'] = lam_sach_1['van_ban_1'].apply(lambda x: replace_emojis_and_words(x, emoji_dict))


lam_sach_1['van_ban_2'] = lam_sach_1['van_ban_2'].apply(remove_punctuation)
lam_sach_1['van_ban_2'] = lam_sach_1['van_ban_2'].apply(lambda x: replace_emojis_and_words(x, td_dict))
lam_sach_1['van_ban_2'] = lam_sach_1['van_ban_2'].apply(normalize_space)

In [ ]:
lam_sach_1[['ma_phan_hoi', 'van_ban_2']].to_csv('Cau2/lam_sach_2.csv', index=False)

In [ ]:
# c
import underthesea

In [ ]:
lam_sach_2 = pd.read_csv('Cau2/lam_sach_2.csv')

In [ ]:
lam_sach_2

In [ ]:
def tokenize(text):
    return underthesea.word_tokenize(text, format='text')

def remove_stop_words(s):
    list = s.split()
    with open('Data/Cau2/tu_dung.txt', 'r', encoding='utf-8') as f:
        stop_words = set(f.read().split('\n'))
    
    tmp =  [i for i in list if i not in stop_words]
    return ' '.join(tmp)

lam_sach_2['chuoi_token'] = lam_sach_2['van_ban_2'].apply(tokenize)
lam_sach_2['chuoi_token_sau_xoa'] = lam_sach_2['chuoi_token'].apply(remove_stop_words)

In [ ]:
for i in range(len(lam_sach_2)):
    lam_sach_2.loc[i, 'so_token'] = len(lam_sach_2.loc[i, 'chuoi_token'].split()) - len(lam_sach_2.loc[i, 'chuoi_token_sau_xoa'].split())

In [ ]:
lam_sach_2[['ma_phan_hoi', 'chuoi_token', 'so_token']].to_csv('Cau2/tokens_ktx.csv', index=False)

In [ ]:
# d
tokens = pd.read_csv('Cau2/tokens_ktx.csv')
tokens

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns

all_words = [word for tokens in tokens['chuoi_token'].str.split() for word in tokens]

word_counts = Counter(all_words)
top10 = pd.DataFrame(word_counts.items(), columns=['tu_khoa', 'so_lan'])
top10 = top10.sort_values(by=['so_lan'], ascending=False).head(10)

top10

In [ ]:
l = lam_sach_2['van_ban_2'].tolist()

for i in range(len(top10)):
    cnt = 0
    if top10.reset_index().loc[i, 'tu_khoa'] in l:
        cnt+=1
    a = top10.reset_index().loc[i, 'so_phan_hoi']=cnt

In [ ]:
top10.to_csv('Cau2/top_tu_khoa.csv')

In [ ]:
# d

plt.figure(figsize=(10,20))
sns.barplot(data=top10, x='tu_khoa', y='so_lan', hue='tu_khoa')
plt.title('Top 10 từ khóa xuất hiện nhiều nhất')
plt.xlabel('Tần suất')
plt.ylabel('Từ vựng')
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig('Cau2/bieu_do_tu_khoa.png')
plt.show()

In [ ]:
from wordcloud import WordCloud

wc = WordCloud(
    width=800,
    height=400,
    background_color='white',
    colormap='plasma'
).generate_from_frequencies(word_counts)


plt.figure(figsize=(12,6))
plt.imshow(wc, interpolation="bilinear")
plt.axis('off')
plt.title('Word cloud tổng thể')
plt.show()

wc.to_file('Cau2/may_tu.png')